# Pull NeurIPs papers utilizing OpenReview API

In [86]:
import pandas as pd
import matplotlib.pyplot as plt
import openreview
from openreview.venue import Venue
import os
from dotenv import load_dotenv

In [87]:
load_dotenv() 

OPENREVIEW_EMAIL = os.getenv("OPENREVIEW_EMAIL")
OPENREVIEW_PASSWORD = os.getenv("OPENREVIEW_PASSWORD")

In [88]:
client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net',
    username=OPENREVIEW_EMAIL,
    password=OPENREVIEW_PASSWORD
)

In [89]:
venue = Venue(client, 'NeurIPS.cc/2025/Conference', 'openreview.net/Support')
accepted = venue.get_submissions(accepted=True)
print(f"Found {len(accepted)} accepted papers")

Found 5286 accepted papers


In [90]:
# full content of first submission
for key, val in accepted[0].content.items():
    print(f"{key}: {val}")

title: {'value': 'Time-o1: Time-Series Forecasting Needs Transformed Label Alignment'}
authors: {'value': ['Hao Wang', 'Licheng Pan', 'Zhichao Chen', 'Xu Chen', 'Qingyang Dai', 'Lei Wang', 'Haoxuan Li', 'Zhouchen Lin']}
authorids: {'value': ['~Hao_Wang28', '~Licheng_Pan1', '~Zhichao_Chen2', '~Xu_Chen13', '~Qingyang_Dai1', '~Lei_Wang46', '~Haoxuan_Li6', '~Zhouchen_Lin1']}
keywords: {'value': ['Time-Series', 'Label Autocorrelation', 'Orthogonalization']}
abstract: {'value': 'Training time-series forecasting models poses unique challenges in loss function design. Most existing approaches adopt temporal mean squared error, but this study reveals two critical limitations: (1) it ignores the presence of label autocorrelation, which biases it from the true label sequence likelihood;  (2) it involves excessive number of tasks, which complicates optimization, especially for long-term forecasting. To address these issues, we introduce Time-o1, a transform-enhanced loss function for time-series f

In [91]:
def clean_venue(venue):
    if venue is None:
        return None
    venue = venue.strip().split()[-1]
    return venue

In [92]:
papers = []
for submission in accepted:
    content = submission.content
    papers.append({
        'title': content['title']['value'].strip(),
        'abstract': content.get('abstract', {}).get('value', None).replace('\n', ' ').strip(),
        'first_author': content.get('authors', {}).get('value', [None])[0],
        'authors': content.get('authors', {}).get('value', []),
        'keywords': content.get('keywords', {}).get('value', []),
        'neurips_status': clean_venue(content.get('venue', {}).get('value', None)),
    })
papers[:3]

[{'title': 'Time-o1: Time-Series Forecasting Needs Transformed Label Alignment',
  'abstract': 'Training time-series forecasting models poses unique challenges in loss function design. Most existing approaches adopt temporal mean squared error, but this study reveals two critical limitations: (1) it ignores the presence of label autocorrelation, which biases it from the true label sequence likelihood;  (2) it involves excessive number of tasks, which complicates optimization, especially for long-term forecasting. To address these issues, we introduce Time-o1, a transform-enhanced loss function for time-series forecasting. The central idea is to transform the label sequence into decorrelated components with discriminated significance. Models are then trained to align the most significant components, thereby effectively mitigating label autocorrelation and reducing task amount. Experiments demonstrate that Time-o1 achieves state-of-the-art performance and is compatible with various forec

In [93]:
df = pd.DataFrame(papers)
print(df.shape)
df

(5286, 6)


,title,abstract,first_author,authors,keywords,neurips_status
0,Time-o1: Time-Series Forecasting Needs Transfo...,Training time-series forecasting models poses ...,Hao Wang,"[Hao Wang, Licheng Pan, Zhichao Chen, Xu Chen,...","[Time-Series, Label Autocorrelation, Orthogona...",poster
1,REVE: A Foundation Model for EEG - Adapting to...,Foundation models have transformed AI by reduc...,Yassine El Ouahidi,"[Yassine El Ouahidi, Jonathan Lys, Philipp Thö...","[Foundation Model, EEG, SSL, BCI]",poster
2,ModHiFi: Identifying High Fidelity predictive ...,"Open weight models, which are ubiquitous, rare...",Dhruva Kashyap,"[Dhruva Kashyap, Chaitanya Murti, Pranav K Nay...","[Pruning, Machine Unlearning]",spotlight
3,The Structure of Relation Decoding Linear Oper...,This paper investigates the structure of linea...,Miranda Anna Christ,"[Miranda Anna Christ, Adrián Csiszárik, Gergel...","[large language models, relations, tensor netw...",spotlight
4,Vulnerable Data-Aware Adversarial Training,Fast adversarial training (FAT) has been consi...,Yuqi Feng,"[Yuqi Feng, Jiahao Fan, Yanan Sun]","[Adversarial Training, Adversarial Robustness,...",poster
...,...,...,...,...,...,...
5281,Differentiable Sparsity via $D$-Gating: Simple...,Structured sparsity regularization offers a pr...,Chris Kolb,"[Chris Kolb, Laetitia Frost, Bernd Bischl, Dav...","[Structured Sparsity, Overparametrization, Gat...",spotlight
5282,Bio-Inspired Image Restoration,"Image restoration aims to recover sharp, high-...",Yuning Cui,"[Yuning Cui, Wenqi Ren, Alois Knoll]","[Image restoration, All-in-one image restorati...",poster
5283,Machine Unlearning via Task Simplex Arithmetic,As foundation Vision-Language Models (VLMs) un...,Junhao Dong,"[Junhao Dong, Hao Zhu, Yifei Zhang, Xinghua Qu...","[Machine Unlearning, Simplex, VLM]",poster
5284,CrossSpectra: Exploiting Cross-Layer Smoothnes...,Parameter-efficient fine-tuning (PEFT) is esse...,Yifei Zhang,"[Yifei Zhang, Hao Zhu, Junhao Dong, Haoran Shi...",[Parameter-efficient fine-tuning (PEFT)],poster


In [94]:
df.to_csv('data/neurips_2025_accepted_papers.csv', index=False)